In [5]:
# ============================================================
# MultiScaleSleepNet-Inspired Architecture (Plain CE, No SupCon,
# No Artifact Weighting) — Architecture-Only Ablation
#
# UPDATED:
#   - Uses the NEW 3-way split (_train_subs.npy / _val_subs.npy /
#     _test_subs.npy) from preprocessed_split_v2. Val is carved
#     out of the original train pool; test is untouched.
#   - Checkpoint selection now uses VAL F1, NOT test F1 (fixes
#     the earlier test-leakage issue).
#   - Test set is evaluated exactly once per seed, after loading
#     the val-selected best checkpoint.
#   - Full per-epoch CSV logging: train loss/acc/f1, val loss/
#     acc/f1/kappa, and per-class val F1 for every epoch of every
#     seed -- so you can regenerate any plot later (loss curves,
#     F1 curves, per-class trends) without re-running training.
#
# Purpose: Isolate the effect of architecture choice from the
# effect of SupCon / artifact-aware training. This script uses
# PLAIN cross-entropy loss (with class weights only) and does
# NOT use artifact quality weighting, so that any accuracy gain
# can be attributed purely to the architecture.
#
# Architecture (inspired by MultiScaleSleepNet, Liu et al. 2025,
# Sensors, DOI: 10.3390/s25206328):
#   1. Parallel multi-scale CNN branches (variable kernel sizes)
#      -> capture fast/slow EEG rhythms at different resolutions
#   2. FFT-based spectral branch -> explicit frequency-domain
#      features alongside raw time-domain convolution
#   3. Squeeze-and-Excitation (SE) block -> adaptive channel
#      recalibration
#   4. BiLSTM -> sequential dependency modeling across the
#      intra-epoch feature sequence
#   5. Transformer encoder layer -> attention-based context
#      (applied at the inter-epoch / context-window level,
#      replacing the BiMamba block used in your other pipeline)
#   6. Plain weighted cross-entropy loss (NO SupCon, NO artifact
#      weighting) -> isolates architecture-only contribution
#
# NOTE: Original MultiScaleSleepNet was evaluated on Sleep-EDF
# (clean clinical PSG), NOT on wearable/Wearanize+ data. This
# script re-implements the core architectural ideas and trains
# them on your Wearanize+ split for a fair, same-data comparison
# against your BiT-MamSleep+SupCon results.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS & CONFIG
# ============================================================
# DATA_PATH  = where the actual per-subject .npz EEG/EOG/label
#              files live (the original preprocessing output).
# SPLIT_PATH = where _train_subs.npy / _val_subs.npy / _test_subs.npy
#              live (this is the NEW folder make_train_val_split.py
#              wrote to -- it only contains the 3 split-list files,
#              NOT the .npz data itself, so it must stay separate
#              from DATA_PATH or the dataset will load 0 samples).
DATA_PATH  = r"D:\22\AA\preprocess\preprocessed_FFinal"
SPLIT_PATH = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"
EVAL_PATH  = r"D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
CONTEXT     = 7
WINDOW      = 2 * CONTEXT + 1     # 15
N_EPOCHS    = 30
BATCH_SIZE  = 64
SEEDS       = [42, 123, 256, 789, 999]

D_MODEL = 128
DROPOUT = 0.4
N_HEADS = 4      # Transformer attention heads

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {device}")
print(f"Architecture : MultiScaleSleepNet-inspired (CNN+SE+BiLSTM+Transformer)")
print(f"Loss         : PLAIN weighted CE  (NO SupCon, NO artifact weighting)")
print(f"Context      : {CONTEXT} (window={WINDOW})")
print(f"Output       : {EVAL_PATH}")


# ============================================================
# FIXED 3-WAY SPLIT (train / val / test — val carved from train,
# test untouched, as created by make_train_val_split.py)
# ============================================================
_train_path = os.path.join(SPLIT_PATH, "_train_subs.npy")
_val_path   = os.path.join(SPLIT_PATH, "_val_subs.npy")
_test_path  = os.path.join(SPLIT_PATH, "_test_subs.npy")

for p, name in [(_train_path, "_train_subs.npy"),
                (_val_path,   "_val_subs.npy"),
                (_test_path,  "_test_subs.npy")]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"{name} not found at {SPLIT_PATH}. Run make_train_val_split.py "
            f"first (or point SPLIT_PATH to the folder that already has it)."
        )

TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
VAL_SUBS   = np.load(_val_path,   allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()

# Hard safety check -- catch any accidental overlap immediately
assert set(TRAIN_SUBS).isdisjoint(VAL_SUBS),  "TRAIN/VAL subject overlap!"
assert set(TRAIN_SUBS).isdisjoint(TEST_SUBS), "TRAIN/TEST subject overlap!"
assert set(VAL_SUBS).isdisjoint(TEST_SUBS),   "VAL/TEST subject overlap!"

print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Val:{len(VAL_SUBS)}  Test:{len(TEST_SUBS)}")


# ============================================================
# DATASET (plain — no artifact weight column used in the loss)
# ============================================================
class PlainDataset(Dataset):
    def __init__(self, subject_list, data_path, context=CONTEXT):
        self.context = context
        self.data  = []
        self.index = []

        label_counter = Counter()
        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                eeg    = d['eeg'][:, [0, 1], :]
                eog    = d['eog'][:, [0], :]
                signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                labels = d['labels'].copy()

            n = len(labels)
            sub_idx = len(self.data)
            self.data.append((signal, labels))
            for i in range(n):
                self.index.append((sub_idx, i, n))
                label_counter[int(labels[i])] += 1

        self.label_counts = np.array(
            [label_counter[i] for i in range(5)], dtype=np.float32
        )
        total = len(self.index)
        ram = sum(s.nbytes for s, _ in self.data) / 1e9
        print(f"  Subjects: {len(self.data)}   Samples: {total:,}   RAM: {ram:.2f} GB")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n = self.index[idx]
        signal, labels = self.data[sub_idx]
        window_epochs = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            window_epochs.append(signal[ei])
        x = np.stack(window_epochs, axis=0)
        y = int(labels[center_i])
        return torch.FloatTensor(x), torch.tensor(y, dtype=torch.long)


print(f"\nLoading .npz signal data from: {DATA_PATH}")
print("Building datasets...")
print("Train:")
train_ds = PlainDataset(TRAIN_SUBS, DATA_PATH)
print("Val:")
val_ds   = PlainDataset(VAL_SUBS, DATA_PATH)
print("Test:")
test_ds  = PlainDataset(TEST_SUBS, DATA_PATH)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    if len(ds) == 0:
        raise RuntimeError(
            f"{name}_ds has 0 samples! This means none of the subject "
            f".npz files were found under DATA_PATH={DATA_PATH}. "
            f"Check that DATA_PATH points to the folder with the actual "
            f"per-subject .npz files (not the split-list-only folder)."
        )
print("Datasets ready.")


# ============================================================
# MULTI-SCALE CNN + SPECTRAL (FFT) BRANCH + SE BLOCK
# ============================================================
class SEBlock(nn.Module):
    """Squeeze-and-Excitation: adaptive channel recalibration."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        # x: (B, C, T)
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    """
    Parallel convolutional branches at different kernel scales
    (fast/slow rhythms) + one FFT-magnitude spectral branch,
    concatenated and recalibrated with SE.
    """
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 4  # 4 branches: small/med/large/spectral

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel,
                          stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(),
                nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(),
                nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)   # fast rhythms (spindle/alpha)
        self.medium = time_branch(50)
        self.large  = time_branch(100)  # slow rhythms (delta)

        # Spectral branch: operates on |FFT(x)| instead of raw x
        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(),
            nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(),
            nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        # x: (B, C, T) raw time-domain signal
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        # spectral branch: magnitude of real FFT along time axis
        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        # pad/truncate spectral input to same length as x for conv
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)  # (B, 4*mid, L)
        feat = self.se(feat)
        return self.proj(feat)                     # (B, d_model, L)


# ============================================================
# BiLSTM + TRANSFORMER ENCODER (replaces BiMamba in this variant)
# ============================================================
class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            d_model, d_model // 2, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, L, d_model)
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)
        attn_out = self.transformer(x)
        return self.norm2(x + attn_out)


# ============================================================
# FULL MODEL: MultiScaleSleepNet-Inspired (plain, no SupCon head)
# ============================================================
class MultiScaleSleepNetPlain(nn.Module):
    def __init__(
        self, in_ch=3, d_model=D_MODEL, n_layers=2,
        dropout=DROPOUT, n_classes=5, context=CONTEXT
    ):
        super().__init__()
        self.context = context
        self.d_model = d_model

        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)
        self.intra_seq_len = self.cnn.out_len

        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.inter_pos = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        # x: (B, W, C, T)
        B, W, C, T = x.shape
        cnn_out = self.cnn(x.view(B * W, C, T))          # (B*W, d_model, L)
        cnn_out = cnn_out.permute(0, 2, 1)                # (B*W, L, d_model)
        intra = self.intra_blocks(cnn_out)
        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)

        inter = epoch_feat + self.inter_pos
        inter = self.inter_blocks(inter)
        center = inter[:, self.context, :]

        logits = self.classifier(center)
        return logits


# ============================================================
# CLASS WEIGHTS -- computed from TRAIN ONLY (unchanged principle)
# ============================================================
cw_np = train_ds.label_counts.sum() / (5 * train_ds.label_counts)
cw = torch.FloatTensor(cw_np).to(device)
print(f"\nClass weights (from TRAIN set only):")
for name, w in zip(LABEL_NAMES, cw_np):
    print(f"  {name}: {w:.3f}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_epoch_fn(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    preds, labs = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(loader)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    return total_loss / n, acc, f1


@torch.no_grad()
def evaluate_fn(model, loader, criterion=None):
    """
    Evaluates on a given loader. If `criterion` is passed, also
    returns the mean loss on that set (used for VAL, so we can log
    val_loss alongside val_acc/val_f1 every epoch). For the final,
    one-time TEST evaluation, criterion can be omitted (loss not
    needed for the headline test report, only acc/f1/kappa/per-class).
    """
    model.eval()
    preds, labs = [], []
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        if criterion is not None:
            loss = criterion(logits, y)
            total_loss += loss.item()
            n_batches += 1
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    preds = np.array(preds); labs = np.array(labs)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs, preds)
    per_cls = f1_score(labs, preds, average=None, zero_division=0, labels=[0, 1, 2, 3, 4])
    mean_loss = (total_loss / n_batches) if n_batches > 0 else None
    return acc, f1, kappa, per_cls, mean_loss


# ============================================================
# CSV SETUP
#
# Two files:
#   1. epoch_log.csv   -- EVERY epoch, EVERY seed: train + val
#                          metrics. This is what you'll use to
#                          regenerate loss curves, F1 curves,
#                          per-class trends, etc. without re-running
#                          training.
#   2. summary.csv      -- ONE row per seed: the FINAL test-set
#                          result (evaluated once, using the
#                          val-selected best checkpoint).
# ============================================================
epoch_csv_path = os.path.join(EVAL_PATH, "epoch_log.csv")
epoch_fields = [
    "seed", "epoch",
    "train_loss", "train_acc", "train_f1_macro",
    "val_loss", "val_acc", "val_f1_macro", "val_kappa",
    "val_f1_Wake", "val_f1_N1", "val_f1_N2", "val_f1_N3", "val_f1_REM",
    "lr", "is_best",
]
with open(epoch_csv_path, 'w', newline='') as f:
    csv.DictWriter(f, epoch_fields).writeheader()

csv_summary_path = os.path.join(EVAL_PATH, "summary.csv")
summary_fields = [
    "seed",
    "best_epoch", "best_val_f1",
    "test_acc", "test_f1_macro", "test_kappa",
    "test_f1_Wake", "test_f1_N1", "test_f1_N2", "test_f1_N3", "test_f1_REM",
]
with open(csv_summary_path, 'w', newline='') as f:
    csv.DictWriter(f, summary_fields).writeheader()

print(f"\nPer-epoch log will be saved to : {epoch_csv_path}")
print(f"Final test summary saved to    : {csv_summary_path}")


# ============================================================
# 5-SEED TRAINING LOOP
# ============================================================
all_results = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(seed)
    )
    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = MultiScaleSleepNetPlain(in_ch=3).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters : {n_params:,}")

    # PLAIN weighted cross-entropy -- no SupCon term added
    criterion = nn.CrossEntropyLoss(weight=cw)

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4,
                             betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_val_f1 = 0.0
    best_epoch  = -1
    best_path   = os.path.join(EVAL_PATH, f"best_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = train_epoch_fn(
            model, train_loader, optimizer, scheduler, criterion
        )

        # === VALIDATION -- used for checkpoint selection ===
        vl_acc, vl_f1, vl_kap, vl_per, vl_loss = evaluate_fn(
            model, val_loader, criterion=criterion
        )

        is_best = 0
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_epoch  = epoch
            torch.save(model.state_dict(), best_path)
            is_best = 1

        lr = optimizer.param_groups[0]['lr']
        tag = " <- BEST" if is_best else ""
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] "
              f"TrLoss:{tr_loss:.3f} TrAcc:{tr_acc:.3f} TrF1:{tr_f1:.3f} | "
              f"ValLoss:{vl_loss:.3f} ValAcc:{vl_acc:.3f} ValF1:{vl_f1:.3f} "
              f"Valk:{vl_kap:.3f} LR:{lr:.2e}{tag}")

        # --- log EVERY epoch, unconditionally, to epoch_log.csv ---
        with open(epoch_csv_path, 'a', newline='') as f:
            csv.DictWriter(f, epoch_fields).writerow({
                "seed": seed, "epoch": epoch,
                "train_loss": round(tr_loss, 6),
                "train_acc": round(tr_acc, 6),
                "train_f1_macro": round(tr_f1, 6),
                "val_loss": round(vl_loss, 6) if vl_loss is not None else "",
                "val_acc": round(vl_acc, 6),
                "val_f1_macro": round(vl_f1, 6),
                "val_kappa": round(vl_kap, 6),
                "val_f1_Wake": round(vl_per[0], 6),
                "val_f1_N1":   round(vl_per[1], 6),
                "val_f1_N2":   round(vl_per[2], 6),
                "val_f1_N3":   round(vl_per[3], 6),
                "val_f1_REM":  round(vl_per[4], 6),
                "lr": lr,
                "is_best": is_best,
            })

    # === FINAL, ONE-TIME TEST EVALUATION (val-selected checkpoint) ===
    model.load_state_dict(torch.load(best_path, map_location=device))
    test_acc, test_f1, test_kap, test_per, _ = evaluate_fn(model, test_loader, criterion=None)

    print(f"\n  Seed {seed}: best val F1={best_val_f1:.4f} at epoch {best_epoch}")
    print(f"  Seed {seed} FINAL TEST (evaluated once): "
          f"Acc={test_acc*100:.2f}% F1={test_f1:.4f} k={test_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {test_per[i]:.4f}")

    all_results.append({
        'seed': seed, 'acc': test_acc, 'f1': test_f1, 'kappa': test_kap,
        'per_cls': test_per, 'best_epoch': best_epoch, 'best_val_f1': best_val_f1
    })

    with open(csv_summary_path, 'a', newline='') as f:
        csv.DictWriter(f, summary_fields).writerow({
            "seed": seed,
            "best_epoch": best_epoch,
            "best_val_f1": round(best_val_f1, 4),
            "test_acc": round(test_acc, 4),
            "test_f1_macro": round(test_f1, 4),
            "test_kappa": round(test_kap, 4),
            "test_f1_Wake": round(test_per[0], 4),
            "test_f1_N1":   round(test_per[1], 4),
            "test_f1_N2":   round(test_per[2], 4),
            "test_f1_N3":   round(test_per[3], 4),
            "test_f1_REM":  round(test_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs = np.array([r['acc'] for r in all_results]) * 100
f1s = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])

print(f"\n{'='*60}\nMULTISCALESLEEPNET-INSPIRED (PLAIN, VAL-SELECTED, HONEST TEST) -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")

print(f"\nAll per-epoch train/val metrics : {epoch_csv_path}")
print(f"Final per-seed test summary     : {csv_summary_path}")
print("Done!")

Device       : cuda
Architecture : MultiScaleSleepNet-inspired (CNN+SE+BiLSTM+Transformer)
Loss         : PLAIN weighted CE  (NO SupCon, NO artifact weighting)
Context      : 7 (window=15)
Output       : D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed
Split loaded -> Train:65  Val:11  Test:20

Loading .npz signal data from: D:\22\AA\preprocess\preprocessed_FFinal
Building datasets...
Train:
  Subjects: 65   Samples: 60,605   RAM: 2.18 GB
Val:
  Subjects: 11   Samples: 10,742   RAM: 0.39 GB
Test:
  Subjects: 20   Samples: 19,763   RAM: 0.71 GB
Datasets ready.

Class weights (from TRAIN set only):
  Wake: 2.130
  N1: 3.183
  N2: 0.432
  N3: 1.008
  REM: 1.098

Per-epoch log will be saved to : D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed\epoch_log.csv
Final test summary saved to    : D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed\summary.csv

SEED 42  (1/5)
  Parameters : 1,013,781
  Ep[01/30] TrLoss:0.861 TrAcc:0.681 TrF1:0.626 | ValLoss:0.614 ValA

In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

# ============================================================
# CONFIG
# ============================================================
DATA_PATH = r"D:\22\AA\preprocess\preprocessed_FFinal"
SPLIT_PATH = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"
CKPT_DIR = r"D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed"
SAVE_DIR = r"D:\22\AA\AA journal\evaluation\ensemble_5seed_multiscale_plain_c7_valfixed"

os.makedirs(SAVE_DIR, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
SEEDS = [42, 123, 256, 789, 999]

CONTEXT = 7
WINDOW = 15
BATCH_SIZE = 64
D_MODEL = 128
DROPOUT = 0.4
N_HEADS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"Checkpoint directory: {CKPT_DIR}")
print(f"Output directory: {SAVE_DIR}")

# ============================================================
# DATASET
# ============================================================
class PlainDataset(Dataset):
    def __init__(self, subject_list, data_path, context=7):
        self.context = context
        self.data = []
        self.index = []

        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                print(f"WARNING: missing {fp}")
                continue

            with np.load(fp) as d:
                eeg = d["eeg"][:, [0, 1], :]
                eog = d["eog"][:, [0], :]
                signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                labels = d["labels"].copy()

            sub_idx = len(self.data)
            n = len(labels)

            self.data.append((signal, labels))

            for i in range(n):
                self.index.append((sub_idx, i, n))

        print(f"Subjects loaded: {len(self.data)}")
        print(f"Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n = self.index[idx]
        signal, labels = self.data[sub_idx]

        window_epochs = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            window_epochs.append(signal[ei])

        x = np.stack(window_epochs, axis=0)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(int(labels[center_i]), dtype=torch.long)
        )

# ============================================================
# LOAD TEST SUBJECTS
# ============================================================
test_path = os.path.join(SPLIT_PATH, "_test_subs.npy")

if not os.path.exists(test_path):
    raise FileNotFoundError(test_path)

TEST_SUBS = np.load(test_path, allow_pickle=True).tolist()

print(f"\nTest subjects: {len(TEST_SUBS)}")

test_ds = PlainDataset(TEST_SUBS, DATA_PATH, CONTEXT)

if len(test_ds) == 0:
    raise RuntimeError("Test dataset is empty.")

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# ============================================================
# MODEL
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=128, dropout=0.4):
        super().__init__()

        mid = d_model // 4

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(
                    in_ch,
                    mid,
                    kernel_size=kernel,
                    stride=6,
                    padding=kernel // 2
                ),
                nn.BatchNorm1d(mid),
                nn.GELU(),
                nn.MaxPool1d(4, 4),

                nn.Conv1d(
                    mid,
                    mid,
                    kernel_size=8,
                    padding=4
                ),
                nn.BatchNorm1d(mid),
                nn.GELU(),
                nn.MaxPool1d(2, 2),

                nn.Dropout(dropout)
            )

        self.small = time_branch(25)
        self.medium = time_branch(50)
        self.large = time_branch(100)

        self.spectral = nn.Sequential(
            nn.Conv1d(
                in_ch,
                mid,
                kernel_size=25,
                stride=6,
                padding=12
            ),
            nn.BatchNorm1d(mid),
            nn.GELU(),
            nn.MaxPool1d(4, 4),

            nn.Conv1d(
                mid,
                mid,
                kernel_size=8,
                padding=4
            ),
            nn.BatchNorm1d(mid),
            nn.GELU(),
            nn.MaxPool1d(2, 2),

            nn.Dropout(dropout)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)

            L_s = self.small(dummy).shape[-1]
            L_m = self.medium(dummy).shape[-1]
            L_l = self.large(dummy).shape[-1]
            L_f = self.spectral(dummy).shape[-1]

        target_L = min(L_s, L_m, L_l, L_f)

        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)

        self.se = SEBlock(4 * mid)

        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)

        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]

        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)
        feat = self.se(feat)

        return self.proj(feat)


class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=4, dropout=0.4):
        super().__init__()

        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.norm1 = nn.LayerNorm(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)

        attn_out = self.transformer(x)

        return self.norm2(x + attn_out)


class MultiScaleSleepNetPlain(nn.Module):
    def __init__(
        self,
        in_ch=3,
        d_model=128,
        n_layers=2,
        dropout=0.4,
        n_classes=5,
        context=7
    ):
        super().__init__()

        self.context = context
        self.d_model = d_model

        self.cnn = MultiScaleCNN(
            in_ch=in_ch,
            d_model=d_model,
            dropout=dropout
        )

        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(
                d_model=d_model,
                n_heads=N_HEADS,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.inter_pos = nn.Parameter(
            torch.randn(1, WINDOW, d_model) * 0.01
        )

        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(
                d_model=d_model,
                n_heads=N_HEADS,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        B, W, C, T = x.shape

        x = x.view(B * W, C, T)

        cnn_out = self.cnn(x).permute(0, 2, 1)

        intra = self.intra_blocks(cnn_out)

        epoch_feat = intra.mean(dim=1)
        epoch_feat = epoch_feat.view(B, W, self.d_model)

        inter = self.inter_blocks(
            epoch_feat + self.inter_pos
        )

        center = inter[:, self.context, :]

        return self.classifier(center)

# ============================================================
# LOAD CHECKPOINTS
# ============================================================
models = []

print("\nLoading checkpoints...")

for seed in SEEDS:
    ckpt_path = os.path.join(
        CKPT_DIR,
        f"best_seed{seed}.pt"
    )

    if not os.path.exists(ckpt_path):
        print(f"WARNING: missing seed {seed}")
        continue

    model = MultiScaleSleepNetPlain(
        in_ch=3,
        d_model=D_MODEL,
        dropout=DROPOUT
    ).to(device)

    checkpoint = torch.load(
        ckpt_path,
        map_location=device
    )

    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]

    model.load_state_dict(checkpoint)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    models.append((seed, model))

    print(f"Loaded seed {seed}")

if not models:
    raise RuntimeError("No checkpoints found.")

print(f"\nTotal models loaded: {len(models)}")

# ============================================================
# EVALUATION
# ============================================================
@torch.no_grad()
def evaluate_single(model, loader):
    preds = []
    labels = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)

        logits = model(x)
        pred = logits.argmax(dim=1)

        preds.extend(pred.cpu().numpy())
        labels.extend(y.numpy())

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )
    kappa = cohen_kappa_score(labels, preds)

    return acc, f1, kappa, preds, labels


@torch.no_grad()
def evaluate_ensemble(models, loader):
    preds = []
    labels = []
    probabilities = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)

        prob_sum = None

        for _, model in models:
            logits = model(x)
            probs = F.softmax(logits, dim=1)

            if prob_sum is None:
                prob_sum = probs
            else:
                prob_sum += probs

        avg_probs = prob_sum / len(models)

        pred = avg_probs.argmax(dim=1)

        preds.extend(pred.cpu().numpy())
        labels.extend(y.numpy())
        probabilities.extend(avg_probs.cpu().numpy())

    preds = np.asarray(preds)
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)

    acc = accuracy_score(labels, preds)

    f1 = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    kappa = cohen_kappa_score(labels, preds)

    per_class_f1 = f1_score(
        labels,
        preds,
        average=None,
        labels=[0, 1, 2, 3, 4],
        zero_division=0
    )

    cm = confusion_matrix(
        labels,
        preds,
        labels=[0, 1, 2, 3, 4]
    )

    return (
        acc,
        f1,
        kappa,
        per_class_f1,
        cm,
        preds,
        labels,
        probabilities
    )

# ============================================================
# INDIVIDUAL RESULTS
# ============================================================
print("\n" + "=" * 70)
print("INDIVIDUAL TEST RESULTS")
print("=" * 70)

individual_rows = []

for seed, model in models:
    acc, f1, kap, preds, labels = evaluate_single(
        model,
        test_loader
    )

    individual_rows.append({
        "seed": seed,
        "accuracy": acc,
        "macro_f1": f1,
        "kappa": kap
    })

    print(
        f"Seed {seed}: "
        f"Acc={acc*100:.2f}% "
        f"F1={f1:.4f} "
        f"Kappa={kap:.4f}"
    )

individual_df = pd.DataFrame(individual_rows)

mean_row = {
    "seed": "Mean",
    "accuracy": individual_df["accuracy"].mean(),
    "macro_f1": individual_df["macro_f1"].mean(),
    "kappa": individual_df["kappa"].mean()
}

individual_df = pd.concat(
    [individual_df, pd.DataFrame([mean_row])],
    ignore_index=True
)

individual_csv = os.path.join(
    SAVE_DIR,
    "individual_results.csv"
)

individual_df.to_csv(
    individual_csv,
    index=False
)

# ============================================================
# ENSEMBLE
# ============================================================
print("\n" + "=" * 70)
print(f"SOFT-VOTE ENSEMBLE ({len(models)} MODELS)")
print("=" * 70)

(
    ens_acc,
    ens_f1,
    ens_kap,
    ens_per_class,
    ens_cm,
    ens_preds,
    ens_labels,
    ens_probs
) = evaluate_ensemble(
    models,
    test_loader
)

print(f"Accuracy    : {ens_acc*100:.2f}%")
print(f"Macro F1    : {ens_f1:.4f}")
print(f"Cohen Kappa : {ens_kap:.4f}")

print("\nPer-class F1:")

for name, score in zip(LABEL_NAMES, ens_per_class):
    print(f"{name:>5}: {score:.4f}")

print("\nConfusion Matrix:")
print(ens_cm)

# ============================================================
# SAVE ENSEMBLE SUMMARY
# ============================================================
summary_df = pd.DataFrame([{
    "ensemble": f"soft_vote_{len(models)}models",
    "n_models": len(models),
    "accuracy": ens_acc,
    "macro_f1": ens_f1,
    "kappa": ens_kap,
    "wake_f1": ens_per_class[0],
    "n1_f1": ens_per_class[1],
    "n2_f1": ens_per_class[2],
    "n3_f1": ens_per_class[3],
    "rem_f1": ens_per_class[4]
}])

summary_csv = os.path.join(
    SAVE_DIR,
    "ensemble_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)

# ============================================================
# SAVE CONFUSION MATRIX
# ============================================================
cm_df = pd.DataFrame(
    ens_cm,
    index=LABEL_NAMES,
    columns=LABEL_NAMES
)

cm_df.index.name = "True"

cm_csv = os.path.join(
    SAVE_DIR,
    "confusion_matrix.csv"
)

cm_df.to_csv(cm_csv)

# ============================================================
# SAVE PREDICTIONS + PROBABILITIES
# ============================================================
prediction_df = pd.DataFrame({
    "true_label": ens_labels,
    "predicted_label": ens_preds,
    "true_class": [
        LABEL_NAMES[i] for i in ens_labels
    ],
    "predicted_class": [
        LABEL_NAMES[i] for i in ens_preds
    ],
    "prob_Wake": ens_probs[:, 0],
    "prob_N1": ens_probs[:, 1],
    "prob_N2": ens_probs[:, 2],
    "prob_N3": ens_probs[:, 3],
    "prob_REM": ens_probs[:, 4]
})

prediction_csv = os.path.join(
    SAVE_DIR,
    "ensemble_predictions.csv"
)

prediction_df.to_csv(
    prediction_csv,
    index=False
)

# ============================================================
# SAVE TEXT REPORT
# ============================================================
report_path = os.path.join(
    SAVE_DIR,
    "ensemble_report.txt"
)

with open(report_path, "w") as f:
    f.write("5-SEED SOFT-VOTE ENSEMBLE REPORT\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Models: {len(models)}\n")
    f.write(
        "Seeds: " +
        ", ".join(str(s) for s, _ in models) +
        "\n\n"
    )

    f.write(f"Accuracy    : {ens_acc*100:.4f}%\n")
    f.write(f"Macro F1    : {ens_f1:.4f}\n")
    f.write(f"Cohen Kappa : {ens_kap:.4f}\n\n")

    f.write("Per-class F1\n")
    f.write("-" * 30 + "\n")

    for name, score in zip(LABEL_NAMES, ens_per_class):
        f.write(f"{name}: {score:.4f}\n")

    f.write("\nConfusion Matrix\n")
    f.write(str(ens_cm))
    f.write("\n")

# ============================================================
# FINAL OUTPUT
# ============================================================
mean_acc = individual_df.iloc[:-1]["accuracy"].mean()

gain = ens_acc - mean_acc

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(f"Individual mean accuracy : {mean_acc*100:.2f}%")
print(f"Ensemble accuracy        : {ens_acc*100:.2f}%")
print(f"Ensemble gain            : {gain*100:+.2f} percentage points")

print("\nSaved files:")
print(f"1. {individual_csv}")
print(f"2. {summary_csv}")
print(f"3. {cm_csv}")
print(f"4. {prediction_csv}")
print(f"5. {report_path}")

print("\nDone.")

Device: cuda
Checkpoint directory: D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed
Output directory: D:\22\AA\AA journal\evaluation\ensemble_5seed_multiscale_plain_c7_valfixed

Test subjects: 20
Subjects loaded: 20
Samples: 19,763

Loading checkpoints...
Loaded seed 42
Loaded seed 123
Loaded seed 256
Loaded seed 789
Loaded seed 999

Total models loaded: 5

INDIVIDUAL TEST RESULTS
Seed 42: Acc=82.89% F1=0.7828 Kappa=0.7595
Seed 123: Acc=83.23% F1=0.7867 Kappa=0.7629
Seed 256: Acc=83.54% F1=0.7905 Kappa=0.7675
Seed 789: Acc=82.61% F1=0.7817 Kappa=0.7549
Seed 999: Acc=83.25% F1=0.7914 Kappa=0.7615

SOFT-VOTE ENSEMBLE (5 MODELS)
Accuracy    : 84.57%
Macro F1    : 0.8039
Cohen Kappa : 0.7810

Per-class F1:
 Wake: 0.8226
   N1: 0.5646
   N2: 0.8753
   N3: 0.8533
  REM: 0.9035

Confusion Matrix:
[[1180  228   10    1   79]
 [ 166  962  162    8   35]
 [  10  727 7919  548  174]
 [   1   36  439 3308   84]
 [  14  122  186   20 3344]]

FINAL RESULT
Individual mean accuracy : 83.10%